In [1]:
import sys
import os
# Adds the parent directory to sys.path
sys.path.append(os.path.abspath('..'))
import pandas as pd
from utils import (data_encoder, hyper_param_opt,
                   outlier_detection_by_clust,
                   weighted_distance_calculation,
                   outlier_detection_by_dist,
                   final_plot)
import pickle


In [2]:
def privgen(data, dataset_name):
    """
    This function exist just for development purposes
    In the actual tool, each of the steps of this functions are individual functions
    which should be ran consequently
    The plots that come out of this functions are those created by the intermediate steps
    and theretically speaking, should help the user to choose the parameters
    """

    # ENCODING
    # This is an auxilary function which is not done by the user
    encoded_data = data_encoder.ordinal_encode_categorical(
        data, dataset_name, save=True)

    # GRID-SEARCH
    hyper_param_opt.hyper_param_search(
        encoded_data,
        percentage_outliers=0.1,
        dataset_name=dataset_name,
        top_n=10,
        eps_values=[0.7, 0.8, 0.9, 1., 1.5, 2],
        min_samples_values=range(30, 50),
        save=True)

    # REMOVE DBSCAN OUTLIERS
    # The arguments of this funciton (now hard-coded) should be choosen by the user, 
    # based on the output of the previous step
    outlier_detection_by_clust.create_dbscan_and_clean_data(encoded_data, eps=2, min_samples=33,
                                                            metric='euclidean', algorithm='auto',
                                                            leaf_size=50, dataset_name=dataset_name, p=2, plot=False)

    # GET DISTRIBUTIONS
    weighted_distance_calculation.calculate_plot_distance(dataset_name)

    # REMOVE DISTANT OUTLIERS
    # The thresholds (now hard-coded) should be choosen by the user, 
    # based on the output of the previous step
    threshold = {'0': 30000, '1': 30000, '2': 30000, '3': 30000, '4': 30000, '5': 15000,
                 '6': 21000, '7': 25000, '8': 12000, '9': 25000, '10': 20000, '11': 20000, '12': 22000}

    outlier_detection_by_dist.filter_and_save_by_distance(
        dataset_name, threshold, plot=False)

    # PLOT
    final_plot.plot_data_points(dataset_name)

    # DECODING
    # This is an auxilary function which is not done by the user
    decoded_data = data_encoder.ordinal_decode_categorical(dataset_name)

    return decoded_data

In [3]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
adult = fetch_ucirepo(id=2) 
  
# data (as pandas dataframes) 
X = adult.data.features 
y = adult.data.targets 


In [4]:
data = X
data['income'] = y

In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             48842 non-null  int64 
 1   workclass       47879 non-null  object
 2   fnlwgt          48842 non-null  int64 
 3   education       48842 non-null  object
 4   education-num   48842 non-null  int64 
 5   marital-status  48842 non-null  object
 6   occupation      47876 non-null  object
 7   relationship    48842 non-null  object
 8   race            48842 non-null  object
 9   sex             48842 non-null  object
 10  capital-gain    48842 non-null  int64 
 11  capital-loss    48842 non-null  int64 
 12  hours-per-week  48842 non-null  int64 
 13  native-country  48568 non-null  object
 14  income          48842 non-null  object
dtypes: int64(6), object(9)
memory usage: 5.6+ MB


## Imputation

In [6]:
from sklearn.impute import SimpleImputer

cols = data.columns
imp = SimpleImputer(strategy="most_frequent")
data = pd.DataFrame(imp.fit_transform(data))
data.columns = cols

In [7]:
data.income.value_counts().index

Index(['<=50K', '<=50K.', '>50K', '>50K.'], dtype='object', name='income')

In [8]:
data.income.replace('<=50K', '<=50K.', inplace=True )
data.income.replace('>50K', '>50K.', inplace=True )
data1 = data[data['income'] == '<=50K.'].sample(500, random_state=42)
data2 = data[data['income'] == '>50K.'].sample(500, random_state=42)
data = pd.concat([data1, data2])
data.reset_index(drop=True, inplace=True)

/var/folders/zh/3166vnf95tnblssyvp6rbhyc0000gn/T/ipykernel_59145/32397451.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data.income.replace('<=50K', '<=50K.', inplace=True )
/var/folders/zh/3166vnf95tnblssyvp6rbhyc0000gn/T/ipykernel_59145/32397451.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values alw

In [9]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             1000 non-null   object
 1   workclass       1000 non-null   object
 2   fnlwgt          1000 non-null   object
 3   education       1000 non-null   object
 4   education-num   1000 non-null   object
 5   marital-status  1000 non-null   object
 6   occupation      1000 non-null   object
 7   relationship    1000 non-null   object
 8   race            1000 non-null   object
 9   sex             1000 non-null   object
 10  capital-gain    1000 non-null   object
 11  capital-loss    1000 non-null   object
 12  hours-per-week  1000 non-null   object
 13  native-country  1000 non-null   object
 14  income          1000 non-null   object
dtypes: object(15)
memory usage: 117.3+ KB


## Encoding

In [10]:
data['age'] = data['age'].astype(int)
data['fnlwgt'] = data['fnlwgt'].astype(int)
data['capital-gain'] = data['capital-gain'].astype(int)
data['capital-loss'] = data['capital-loss'].astype(int)
data['hours-per-week'] = data['hours-per-week'].astype(int)

In [11]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             1000 non-null   int64 
 1   workclass       1000 non-null   object
 2   fnlwgt          1000 non-null   int64 
 3   education       1000 non-null   object
 4   education-num   1000 non-null   object
 5   marital-status  1000 non-null   object
 6   occupation      1000 non-null   object
 7   relationship    1000 non-null   object
 8   race            1000 non-null   object
 9   sex             1000 non-null   object
 10  capital-gain    1000 non-null   int64 
 11  capital-loss    1000 non-null   int64 
 12  hours-per-week  1000 non-null   int64 
 13  native-country  1000 non-null   object
 14  income          1000 non-null   object
dtypes: int64(5), object(10)
memory usage: 117.3+ KB


In [12]:
dataset_name= 'adult'
encoded_data = data_encoder.one_hot_encode_categorical(
        data, dataset_name, save=True)


## DBSCAN HyperParameter Optimization

In [ ]:

    # GRID-SEARCH
# hyper_param_opt.hyper_param_search(
#     encoded_data,
#     percentage_outliers=0.1,
#     dataset_name=dataset_name,
#     top_n=10,
#     eps_values=[0.7, 0.8, 0.9, 1., 1.5, 2],
#     min_samples_values=range(30, 50),
#     save=True)

# THIS IS DONE ON AWS

In [ ]:
encoded_data.info()

In [13]:
grid_results = pd.read_csv('../data/adult_grid_search_results.csv')

In [14]:
grid_results


,num_clusters,percentage_outliers,eps,min_samples,metric,algorithm,leaf_size,p,silhouette_score
0,0,1.0,0.7,30,euclidean,auto,10,NaN,-1
1,0,1.0,0.7,30,euclidean,auto,10,1.0,-1
2,0,1.0,0.7,30,euclidean,auto,10,2.0,-1
3,0,1.0,0.7,30,euclidean,auto,20,NaN,-1
4,0,1.0,0.7,30,euclidean,auto,20,1.0,-1
...,...,...,...,...,...,...,...,...,...
14395,0,1.0,2.0,49,manhattan,brute,40,1.0,-1
14396,0,1.0,2.0,49,manhattan,brute,40,2.0,-1
14397,0,1.0,2.0,49,manhattan,brute,50,NaN,-1
14398,0,1.0,2.0,49,manhattan,brute,50,1.0,-1


In [15]:
grid_results.describe()

,num_clusters,percentage_outliers,eps,min_samples,leaf_size,p,silhouette_score
count,14400.0,14400.0,14400.000000,14400.000000,14400.000000,9600.000000,14400.0
mean,0.0,1.0,1.150000,39.500000,30.000000,1.500000,-1.0
std,0.0,0.0,0.457363,5.766482,14.142627,0.500026,0.0
min,0.0,1.0,0.700000,30.000000,10.000000,1.000000,-1.0
25%,0.0,1.0,0.800000,34.750000,20.000000,1.000000,-1.0
50%,0.0,1.0,0.950000,39.500000,30.000000,1.500000,-1.0
75%,0.0,1.0,1.500000,44.250000,40.000000,2.000000,-1.0
max,0.0,1.0,2.000000,49.000000,50.000000,2.000000,-1.0


In [ ]:
grid_results[grid_results.num_clusters == 2].sort_values('silhouette_score', ascending=False)

## REMOVE DBSCAN OUTLIERS



In [ ]:
outlier_detection_by_clust.create_dbscan_and_clean_data(encoded_data, eps=2, min_samples=31,
                                                        metric='euclidean', algorithm='ball_tree',
                                                        leaf_size=50, dataset_name=dataset_name, p=1, plot=False)



## GET DISTRIBUTIONS


In [ ]:
weighted_distance_calculation.calculate_plot_distance(dataset_name)



## REMOVE DISTANT OUTLIERS


In [ ]:
threshold = {'0': 20, '1': 18}

outlier_detection_by_dist.filter_and_save_by_distance(
    dataset_name, threshold, plot=False)



## PLOT


In [ ]:
final_plot.plot_data_points(dataset_name)



## DECODING


In [ ]:
decoded_data = data_encoder.ordinal_decode_categorical(dataset_name)